[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/github-actions-certified/notebooks/day-04-matrix-and-caching.ipynb#scrollTo=a1b2c3d4)

---
# Day 4 · Matrix Strategies and Dependency Caching
**certified-journeys / github-actions-certified** · Day 4 · Workflow Optimization

> **Goal for today:** Understand and write GitHub Actions matrix strategies to run parallel jobs across multiple Python versions and OS targets, and add pip caching to slash CI runtime on repeat runs.


In [ ]:
%pip install -q pyyaml


## Step 1 · Matrix Strategy Basics

A **matrix strategy** tells GitHub Actions to fan out a single job definition into multiple parallel jobs — one per combination of matrix values. You define axes (e.g. `python-version`, `os`) and Actions generates the Cartesian product.

| Concept | What it does |
|---|---|
| `strategy.matrix` | Declares axes and their value lists |
| `${{ matrix.python-version }}` | Interpolates the current cell's value |
| `fail-fast` | If `true` (default), cancels remaining jobs when one fails |
| `max-parallel` | Caps how many jobs run concurrently |

A 2-OS × 3-Python matrix produces **6 parallel jobs** automatically.


In [ ]:
import yaml

# Build a matrix workflow as a Python dict, then render it as YAML
matrix_workflow = {
    "name": "Python matrix CI",
    "on": {"push": {"branches": ["main"]}, "pull_request": {}},
    "jobs": {
        "test": {
            "runs-on": "${{ matrix.os }}",
            "strategy": {
                "matrix": {
                    "python-version": ["3.10", "3.11", "3.12"],
                    "os": ["ubuntu-latest", "macos-latest"],
                }
            },
            "steps": [
                {"uses": "actions/checkout@v4"},
                {
                    "name": "Set up Python ${{ matrix.python-version }}",
                    "uses": "actions/setup-python@v5",
                    "with": {"python-version": "${{ matrix.python-version }}"},
                },
                {
                    "name": "Install dependencies",
                    "run": "pip install -r requirements.txt",
                },
                {
                    "name": "Run tests",
                    "run": "pytest tests/",
                },
            ],
        }
    },
}

print(yaml.dump(matrix_workflow, sort_keys=False, allow_unicode=True))


**What just happened?**

- We defined a matrix with **2 OS values × 3 Python versions = 6 parallel jobs**.
- `runs-on: ${{ matrix.os }}` means each job picks its runner from the matrix row.
- **`actions/setup-python@v5`** interpolates the correct version per job cell.
- Without any extra config, Actions fans out all 6 jobs immediately.


## Step 2 · Counting and Verifying Matrix Jobs

Before you push, you can reason about your matrix locally: count all combinations using Python's `itertools.product`. This is exactly what Actions does when it schedules jobs.

| Setting | Effect |
|---|---|
| `fail-fast: true` | First failure cancels all remaining jobs (default) |
| `fail-fast: false` | All jobs run to completion regardless of failures |
| `max-parallel: N` | At most N jobs run at the same time |


In [ ]:
import itertools

# Simulate the matrix expansion Actions performs
python_versions = ["3.10", "3.11", "3.12"]
operating_systems = ["ubuntu-latest", "macos-latest"]

combinations = list(itertools.product(python_versions, operating_systems))

print(f"Total parallel jobs: {len(combinations)}\n")
for i, (py, os_) in enumerate(combinations, 1):
    print(f"  Job {i:02d}: python={py}  os={os_}")


**What just happened?**

- `itertools.product` mirrors exactly what GitHub Actions computes: the Cartesian product of all matrix axes.
- **6 jobs** fire in parallel (subject to your repo's concurrency limit).
- This is useful for double-checking before pushing — large matrices can exhaust Actions minutes quickly.


## Step 3 · `fail-fast: false` and `max-parallel`

`fail-fast: false` is critical for cross-platform testing: you want to see **all** failures, not just the first. `max-parallel: 3` is useful when you have many matrix cells but want to avoid exhausting API rate limits or parallel job quotas simultaneously.

| Scenario | Recommended setting |
|---|---|
| Debug flaky cross-platform failures | `fail-fast: false` |
| Protect external rate-limited APIs | `max-parallel: 2` or `3` |
| Fast feedback on PRs | `fail-fast: true` (default) |


In [ ]:
# Demonstrate how fail-fast and max-parallel change the strategy block

def build_strategy(fail_fast: bool, max_parallel: int | None = None) -> dict:
    strategy = {
        "fail-fast": fail_fast,
        "matrix": {
            "python-version": ["3.10", "3.11", "3.12"],
            "os": ["ubuntu-latest", "macos-latest"],
        },
    }
    if max_parallel is not None:
        strategy["max-parallel"] = max_parallel
    return strategy


# Default: fail-fast=true, no max-parallel
print("=== Default (fail-fast=true) ===")
print(yaml.dump(build_strategy(fail_fast=True), sort_keys=False))

# Recommended for debugging: see all platform failures
print("=== Debug mode (fail-fast=false, max-parallel=3) ===")
print(yaml.dump(build_strategy(fail_fast=False, max_parallel=3), sort_keys=False))


**What just happened?**

- With `fail-fast: false`, all 6 jobs run even if Job 1 fails — **you see all platform failures at once**.
- `max-parallel: 3` throttles concurrent jobs: the second batch (jobs 4–6) waits for the first 3 to finish.
- The matrix definition itself doesn't change — only execution behaviour.


## Step 4 · Dependency Caching with `actions/cache`

Pip installs the same packages every run unless you cache `~/.cache/pip`. The cache key must be **deterministic**: it must change exactly when dependencies change (i.e. when `requirements.txt` changes) and stay the same otherwise.

**Cache key anatomy:**
```
runner.os-pip-<hash of requirements.txt>
```

- `runner.os` — prevents macOS and Ubuntu from sharing a cache (binaries differ)
- `pip` — namespaces the key from other caches in the same repo
- `hashFiles('**/requirements.txt')` — changes only when the file changes

`restore-keys` provides a **fallback**: if the exact key misses, Actions restores the most recent partial match. This means a new dependency gets installed on top of the old cache — faster than a cold install.


In [ ]:
# Full workflow with matrix + caching
cached_matrix_workflow = {
    "name": "Python matrix CI with caching",
    "on": {"push": {"branches": ["main"]}, "pull_request": {}},
    "jobs": {
        "test": {
            "runs-on": "${{ matrix.os }}",
            "strategy": {
                "fail-fast": False,
                "max-parallel": 3,
                "matrix": {
                    "python-version": ["3.10", "3.11", "3.12"],
                    "os": ["ubuntu-latest", "macos-latest"],
                },
            },
            "steps": [
                {"uses": "actions/checkout@v4"},
                {
                    "name": "Set up Python ${{ matrix.python-version }}",
                    "uses": "actions/setup-python@v5",
                    "with": {"python-version": "${{ matrix.python-version }}"},
                },
                {
                    # Cache pip's download cache — keyed on OS + requirements hash
                    "name": "Cache pip dependencies",
                    "uses": "actions/cache@v4",
                    "with": {
                        "path": "~/.cache/pip",
                        # Key changes only when requirements.txt changes
                        "key": "${{ runner.os }}-pip-${{ hashFiles('**/requirements.txt') }}",
                        # Fallback: restore most recent cache for this OS
                        "restore-keys": "${{ runner.os }}-pip-",
                    },
                },
                {
                    "name": "Install dependencies",
                    "run": "pip install -r requirements.txt",
                },
                {
                    "name": "Run tests",
                    "run": "pytest tests/ -v",
                },
            ],
        }
    },
}

print(yaml.dump(cached_matrix_workflow, sort_keys=False, allow_unicode=True))


**What just happened?**

- The cache step runs **before** `pip install`, so on a cache hit, pip skips downloading packages.
- `hashFiles('**/requirements.txt')` is computed by Actions at workflow start — it's a SHA-256 of the file contents.
- **On a cache hit** the `pip install` step still runs but exits immediately (all wheels already present).
- Each OS gets its own cache partition because `runner.os` is part of the key.


## Step 5 · Simulating Cache Hit vs Miss Logic

In a real Actions run you see `Cache hit` or `Cache miss` in the step log. We can simulate that logic locally by hashing a requirements file and checking if a stored key matches.

This is exactly what `actions/cache` does: it computes the key, looks for a match in the cache store, and reports hit or miss.


In [ ]:
import hashlib
import os
import tempfile


def compute_cache_key(runner_os: str, requirements_content: str) -> str:
    """Mirror hashFiles('**/requirements.txt') for one file."""
    sha = hashlib.sha256(requirements_content.encode()).hexdigest()
    return f"{runner_os}-pip-{sha}"


def simulate_cache_lookup(key: str, store: dict) -> tuple[bool, str | None]:
    """Return (hit, restored_key). restore-keys prefix fallback included."""
    if key in store:
        return True, key
    # Fallback: find most recent partial-match key
    prefix = key.rsplit("-", 1)[0] + "-"   # e.g. "Linux-pip-"
    matches = [k for k in store if k.startswith(prefix)]
    if matches:
        return False, matches[-1]  # partial hit — stale cache restored
    return False, None


# Simulate two runs
reqs_v1 = "scikit-learn==1.4.0\npandas==2.2.0\npytest==8.1.0\n"
reqs_v2 = "scikit-learn==1.5.0\npandas==2.2.0\npytest==8.1.0\n"  # version bump

runner_os = "Linux"
key_run1 = compute_cache_key(runner_os, reqs_v1)
key_run2 = compute_cache_key(runner_os, reqs_v2)

# Pretend run 1 already populated the cache store
cache_store = {key_run1: "cached_wheels_v1"}

print("Run 1 key:", key_run1)
hit1, restored1 = simulate_cache_lookup(key_run1, cache_store)
print(f"  → {'CACHE HIT' if hit1 else 'CACHE MISS'}  (restored: {restored1})\n")

print("Run 2 key (requirements changed):", key_run2)
hit2, restored2 = simulate_cache_lookup(key_run2, cache_store)
print(f"  → {'CACHE HIT' if hit2 else 'CACHE MISS'}  (restored from fallback: {restored2})")
print("  Partial restore means pip downloads only the changed package (scikit-learn).")


**What just happened?**

- Run 1: **exact cache hit** — pip skips all downloads, CI is fast.
- Run 2: requirements changed so the exact key misses, but `restore-keys` finds the previous cache.
- **Partial hit** means pip only downloads the new/changed package, not the entire dependency tree.
- This is why `restore-keys` is worth adding: it prevents cold installs on every dependency bump.


## Step 6 · Matrix with `include` / `exclude` for Edge Cases

Sometimes you want to add a one-off combination (e.g. Windows only for Python 3.12) or drop a known-broken pair. Actions supports `include` and `exclude` within the matrix for exactly this.

| Key | Effect |
|---|---|
| `include` | Adds extra cells to the matrix (or extends existing cells with extra keys) |
| `exclude` | Drops specific cells that would otherwise be generated |


In [ ]:
# Advanced matrix: add a Windows job for 3.12 only, drop macOS+3.10 (known flaky)
advanced_matrix = {
    "strategy": {
        "fail-fast": False,
        "matrix": {
            "python-version": ["3.10", "3.11", "3.12"],
            "os": ["ubuntu-latest", "macos-latest"],
            # include: add a cell that doesn't exist in the base product
            "include": [
                {"python-version": "3.12", "os": "windows-latest"}
            ],
            # exclude: drop a specific combination
            "exclude": [
                {"python-version": "3.10", "os": "macos-latest"}
            ],
        },
    }
}

print(yaml.dump(advanced_matrix, sort_keys=False))

# Compute the effective job count manually
base = list(itertools.product(["3.10", "3.11", "3.12"], ["ubuntu-latest", "macos-latest"]))
excluded = {("3.10", "macos-latest")}
effective = [c for c in base if c not in excluded]
effective.append(("3.12", "windows-latest"))  # from include
print(f"Effective job count: {len(effective)}")
for py, os_ in effective:
    print(f"  python={py}  os={os_}")


**What just happened?**

- Base product: 6 jobs. Minus 1 excluded + 1 included = **6 jobs** with a different composition.
- `include` is additive — it adds cells that don't exist in the base product, or augments existing cells with extra context variables.
- `exclude` lets you skip combinations that are known to be broken without removing the entire axis.


## Step 7 · Putting It All Together — Complete Matrix + Cache Workflow

Here is the production-ready version combining everything from today: matrix across 3 Python versions + 2 OS, `fail-fast: false`, `max-parallel: 3`, and pip caching with `restore-keys`.


In [ ]:
# Production-ready full workflow — save to .github/workflows/ci.yml in your repo
production_workflow = """\
name: CI — Matrix + Cache

on:
  push:
    branches: [main]
  pull_request:

jobs:
  test:
    runs-on: ${{ matrix.os }}
    strategy:
      fail-fast: false       # see ALL failures across all matrix cells
      max-parallel: 3        # at most 3 concurrent jobs
      matrix:
        python-version: ['3.10', '3.11', '3.12']
        os: [ubuntu-latest, macos-latest]

    steps:
      - uses: actions/checkout@v4

      - name: Set up Python ${{ matrix.python-version }}
        uses: actions/setup-python@v5
        with:
          python-version: ${{ matrix.python-version }}

      - name: Cache pip
        uses: actions/cache@v4
        with:
          path: ~/.cache/pip
          key: ${{ runner.os }}-pip-${{ hashFiles('**/requirements.txt') }}
          restore-keys: |
            ${{ runner.os }}-pip-

      - name: Install dependencies
        run: pip install -r requirements.txt

      - name: Run tests
        run: pytest tests/ -v --tb=short
"""

print(production_workflow)

# Parse back to verify it's valid YAML
parsed = yaml.safe_load(production_workflow)
matrix = parsed["jobs"]["test"]["strategy"]["matrix"]
combos = list(itertools.product(matrix["python-version"], matrix["os"]))
print(f"✓ Valid YAML — {len(combos)} matrix jobs defined")


**What just happened?**

- We composed the full workflow as a multiline string — this is what you'd save to `.github/workflows/ci.yml`.
- `yaml.safe_load` confirms the YAML is syntactically valid before you commit it.
- `restore-keys` uses the YAML block scalar (`|`) so the fallback prefix is on its own line — required by Actions.
- **6 parallel jobs**, throttled to 3 at a time, with pip cache acceleration on repeat runs.


## Step 8 · Cache Key Design Workshop

Different project types need different cache paths and key patterns. Here's a reference table and a generator function to help you reason about cache strategy for any tool.

| Tool | Cache path | Key ingredient |
|---|---|---|
| pip | `~/.cache/pip` | `hashFiles('**/requirements*.txt')` |
| npm | `~/.npm` | `hashFiles('**/package-lock.json')` |
| yarn | `.yarn/cache` | `hashFiles('**/yarn.lock')` |
| Maven | `~/.m2/repository` | `hashFiles('**/pom.xml')` |
| Gradle | `~/.gradle/caches` | `hashFiles('**/*.gradle*')` |


In [ ]:
def generate_cache_step(tool: str, runner_os_expr: str = "${{ runner.os }}") -> dict:
    """Generate an actions/cache step for common package managers."""
    configs = {
        "pip": {
            "path": "~/.cache/pip",
            "lock": "**/requirements*.txt",
        },
        "npm": {
            "path": "~/.npm",
            "lock": "**/package-lock.json",
        },
        "yarn": {
            "path": ".yarn/cache",
            "lock": "**/yarn.lock",
        },
        "maven": {
            "path": "~/.m2/repository",
            "lock": "**/pom.xml",
        },
    }
    cfg = configs[tool]
    return {
        "name": f"Cache {tool} dependencies",
        "uses": "actions/cache@v4",
        "with": {
            "path": cfg["path"],
            "key": f"{runner_os_expr}-{tool}-${{{{ hashFiles('{cfg['lock']}') }}}}",
            "restore-keys": f"{runner_os_expr}-{tool}-",
        },
    }


for tool in ["pip", "npm", "yarn", "maven"]:
    step = generate_cache_step(tool)
    print(f"--- {tool} cache step ---")
    print(yaml.dump(step, sort_keys=False))


**What just happened?**

- We built a reusable function that generates correct `actions/cache` steps for four common package managers.
- The key pattern `runner.os-tool-hashFiles` is idiomatic and widely used in the Actions ecosystem.
- **Always namespace with the tool name** — if you cache both pip and npm in the same repo, they must have distinct prefixes.


In [ ]:
# Challenge: Build a matrix workflow for a data science project
#
# Requirements:
#   1. Matrix: Python 3.10, 3.11, 3.12 on ubuntu-latest only
#   2. Add fail-fast: false and max-parallel: 2
#   3. Cache pip with hashFiles on 'requirements.txt'
#   4. Steps: checkout → setup-python → cache → install → pytest
#   5. BONUS: Add an exclude to skip Python 3.10 on this matrix
#             (hint: exclude works even with a single-OS matrix)
#
# Your solution here:

challenge_workflow = {
    "name": "Data science CI",
    "on": {},        # TODO: trigger on push to main
    "jobs": {
        "test": {
            "runs-on": "",  # TODO: fill in the runner
            "strategy": {
                # TODO: matrix, fail-fast, max-parallel
            },
            "steps": [
                # TODO: checkout, setup-python, cache, install, pytest
            ],
        }
    },
}

print(yaml.dump(challenge_workflow, sort_keys=False))


---
## Day 4 key concepts recap

| Concept | What to remember |
|---|---|
| Matrix strategy | Cartesian product of all axes → parallel jobs automatically |
| `fail-fast: false` | See all platform failures; don't cancel on first failure |
| `max-parallel` | Cap concurrent jobs to protect rate-limited dependencies |
| Cache key | Must be deterministic: `runner.os-tool-hashFiles(lockfile)` |
| `restore-keys` | Partial-hit fallback — speeds installs when deps change |
| `include` / `exclude` | Add or drop specific matrix cells without restructuring axes |

> **Tip:** Cache keys must be deterministic and specific. A key like `runner.os-pip-hashFiles` adds a `restore-keys:` fallback for partial hits when requirements change.

---
## What's next
**Day 5** → Secrets management and security best practices — storing credentials, scoping `GITHUB_TOKEN`, and OIDC for keyless cloud auth.

Mark Day 4 complete in your [tracker](../index.html).
